# 02 — Candle Primitives

## Goal
Compute the per-candle features that every downstream step depends on.

## Equations

**Range** — the full price span of a candle:
$$\text{range} = \text{high} - \text{low}$$

**Body** — the absolute distance between open and close:
$$\text{body} = |\text{close} - \text{open}|$$

**Body Ratio** — fraction of the range that is "real move" vs shadow:
$$\text{body\_ratio} = \frac{\text{body}}{\text{range}}$$

- $\text{body\_ratio} \approx 1.0$ → strong directional candle (almost no wicks)
- $\text{body\_ratio} \approx 0.0$ → doji or wide-shadow candle

## Classification Rules

| Flag | Condition | Meaning |
|------|-----------|---------|
| `is_base` | $\text{body\_ratio} \leq 0.50$ | Indecisive / consolidation candle |
| `is_doji` | $\text{body\_ratio} \leq 0.10$ | Near-zero body (perfect indecision) |
| `is_bullish` | $\text{close} > \text{open}$ | Buyers won the session |

## Visualization
The chart below shows candles color-coded by type:
- **Teal** = bullish, **Red** = bearish
- **Grey** = base candle ($\text{body\_ratio} \leq 0.50$)
- **Gold** = doji ($\text{body\_ratio} \leq 0.10$)

The lower panel plots `body_ratio` per candle — the dashed line marks the base threshold at 0.50.


In [1]:
import pandas as pd


BASE_BODY_RATIO_MAX: float = 0.50   # candle is "base" if body / range <= 0.50

df = pd.read_csv("../fixtures.csv", index_col=0, parse_dates=True)

df["range"] = df["high"] - df["low"]
df["body"] = abs(df["close"] - df["open"])
df["body-ratio"] = df["body"] / df["range"]
df["is_base"] = df["body-ratio"] <= BASE_BODY_RATIO_MAX
df["prev-close"] = df["close"].shift(1)
df["is_doji"] = df["body-ratio"] <= 0.1 
df["is_bullish"] = df["close"] > df["open"]
df.to_csv("../fixtures_enhanced.csv")
df.head(20)


,open,high,low,close,volume,range,body,body-ratio,is_base,prev-close,is_doji,is_bullish
Date,,,,,,,,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1000000,1.6,0.6,0.375000,True,NaN,False,True
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1000000,1.5,0.5,0.333333,True,100.6,False,False
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1000000,1.6,0.6,0.375000,True,100.1,False,True
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1000000,1.5,0.5,0.333333,True,100.7,False,False
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1000000,1.6,0.6,0.375000,True,100.2,False,True
2024-02-05 00:00:00+00:00,100.8,101.3,99.8,100.3,1000000,1.5,0.5,0.333333,True,100.8,False,False
2024-02-12 00:00:00+00:00,100.3,101.4,99.8,100.9,1000000,1.6,0.6,0.375000,True,100.3,False,True
2024-02-19 00:00:00+00:00,100.9,101.4,99.9,100.4,1000000,1.5,0.5,0.333333,True,100.9,False,False
2024-02-26 00:00:00+00:00,100.4,101.5,99.9,101.0,1000000,1.6,0.6,0.375000,True,100.4,False,True


In [2]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
COLOR_BASE = "#b0bec5"
COLOR_DOJI = "#ffd700"
BG         = "#131722"
GRID       = "#1e222d"

def candle_color(row):
    if row["is_doji"]: return COLOR_DOJI
    if row["is_base"]: return COLOR_BASE
    return COLOR_BULL if row["is_bullish"] else COLOR_BEAR

colors = df.apply(candle_color, axis=1)

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.03,
)

# ── row 1 : candlesticks ───────────────────────────────────────────────────
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
), row=1, col=1)

# ── row 2 : body-ratio bars ────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=df.index,
    y=df["body-ratio"],
    marker_color=colors,
    name="Body Ratio",
    showlegend=False,
), row=2, col=1)

# threshold line on body-ratio panel
fig.add_hline(
    y=BASE_BODY_RATIO_MAX,
    line_dash="dot", line_color="#888", line_width=1,
    annotation_text=f"base ≤ {BASE_BODY_RATIO_MAX}",
    annotation_font_color="#888",
    annotation_position="bottom right",
    row=2, col=1,
)

# ── layout ─────────────────────────────────────────────────────────────────
fig.update_layout(
    title="Candle Primitives — base, doji & body ratio",
    height=600,
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    legend=dict(orientation="h", y=1.04),
    bargap=0.15,
)

# style both panels
for axis in ["xaxis", "xaxis2", "yaxis", "yaxis2"]:
    fig.update_layout(**{axis: dict(
        gridcolor=GRID,
        showgrid=True,
        zeroline=False,
    )})

fig.update_yaxes(title_text="Price",      row=1, col=1)
fig.update_yaxes(title_text="Body Ratio", row=2, col=1, range=[0, 1.1])

fig.show()
